# SST-2 Competition Exercise: DistilBERT Under Interface Pressure

**Timebox:** 2 hours  
**Mode:** prompt-only; no worked solution is included  
**Primary skill:** reduce an unfamiliar tokenizer/model API to a verified input/output contract


## Scenario

You receive three local SST-2 parquet splits and the name of an approved pretrained checkpoint:

`distilbert/distilbert-base-uncased-finetuned-sst-2-english`

Your job is to build a reliable sentiment pipeline and produce predictions plus short token rationales. The hidden evaluator rewards both classification quality and whether the selected rationale tokens actually influence the model's prediction.

This exercise mirrors a common olympiad pattern: supplied data, a supplied pretrained model family, an unfamiliar structured API, a strict output contract, and limited time.


## Provided Files

All files are in the same directory as this notebook:

- `train.parquet`: labeled training rows.
- `validation.parquet`: labeled validation rows.
- `test.parquet`: test rows; do not assume labels are usable.
- `../06_distilbert_sst2/model_card.md`: downloaded model card.
- `../06_distilbert_sst2/config.json`: downloaded architecture and label configuration.

Expected dataset columns: `idx`, `sentence`, `label`.


## Rules

1. Use only the provided SST-2 data.
2. The approved Transformer checkpoint is the only pretrained language model allowed.
3. Maximum tokenized length: 128.
4. A prediction must be generated for every row in the requested split.
5. Rationale indices must refer to non-padding, non-special tokens from the tokenizer output.
6. Do not begin full training until a two-sentence forward pass has been inspected successfully.

You may use the checkpoint as-is, freeze it and train a small component, or fine-tune it. Prefer the simplest route that produces a verified artifact inside the timebox.


## Scoring Proxy

Use this local proxy to compare attempts:

$$\text{proxy} = 0.75 \cdot \text{validation accuracy} + 0.25 \cdot \text{faithfulness}$$

For each validation example, faithfulness is the decrease in the originally predicted class probability after masking your selected rationale tokens. Clip each decrease to `[0, 1]`, then average it.

This is a practice proxy, not an official CEOAI metric. Its purpose is to force you to keep prediction quality and explanation quality separate, as recommended by ERASER.


## Part 0 - Environment and Reproducibility

The setup cell is provided. Do not add model-specific code here.


In [1]:
from pathlib import Path
import inspect
import json
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
MAX_LENGTH = 128
BATCH_SIZE = 16

DATA_DIR = Path.cwd()
if not (DATA_DIR / "train.parquet").exists():
    DATA_DIR = Path("olympiads/recommended_materials_2026/07_sst2_dataset")

print("device:", DEVICE)
print("data directory:", DATA_DIR.resolve())


device: cuda
data directory: D:\projects\Supervised-Learning-Experiments\olympiads\recommended_materials_2026\07_sst2_dataset


## Part 1 - Inspect the Data Contract

Before touching the model:

1. Load all three parquet files.
2. Print split shapes, column names, dtypes and label values.
3. Verify `idx` uniqueness within each split.
4. Display five short and five long training sentences.
5. Record whether the test labels are usable.

Stop if the schema differs from your assumptions.


In [2]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
validation_df = pd.read_parquet(DATA_DIR / "validation.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")

for name, frame in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}.items():
    print(f"{name}: shape={frame.shape}, columns={frame.columns.tolist()}")
    print(frame.dtypes)
    print("unique labels:", sorted(frame["label"].unique().tolist()))
    print("idx unique:", frame["idx"].is_unique)
    print()

# TODO: display five shortest and five longest training sentences.


train: shape=(67349, 3), columns=['idx', 'sentence', 'label']
idx         int32
sentence      str
label       int64
dtype: object
unique labels: [0, 1]
idx unique: True

validation: shape=(872, 3), columns=['idx', 'sentence', 'label']
idx         int32
sentence      str
label       int64
dtype: object
unique labels: [0, 1]
idx unique: True

test: shape=(1821, 3), columns=['idx', 'sentence', 'label']
idx         int32
sentence      str
label       int64
dtype: object
unique labels: [-1]
idx unique: True



## Part 2 - The 15-Minute Model API Probe

Load the tokenizer and sequence-classification model. Then use exactly two sentences to record:

- tokenizer class and model class;
- `model.config.id2label` and `label2id`;
- `inspect.signature(model.forward)`;
- tokenizer output keys, shapes and dtypes;
- model output keys and logits shape;
- predicted label and confidence for both sentences;
- total and trainable parameter counts.

Expected invariant: logits have shape `(2, 2)`. Do not create a Dataset or DataLoader until this passes.


In [3]:
# TODO: load tokenizer and model, then complete the two-sentence API probe.


## Part 3 - Build the Batched Prediction Path

Create the smallest reusable batching path that:

1. accepts raw sentences;
2. tokenizes with truncation, padding and `MAX_LENGTH`;
3. moves every returned tensor to `DEVICE`;
4. runs inference without gradients;
5. returns class-1 probabilities and predicted labels in original row order.

Self-check on validation:

- number of predictions equals number of rows;
- every predicted label is `0` or `1`;
- every probability is finite and within `[0, 1]`;
- validation accuracy is printed;
- ten errors are displayed with text, true label, prediction and confidence.


In [4]:
# TODO: implement the batched prediction path and evaluate it on validation_df.


## Part 4 - Optional Adaptation

Only attempt adaptation after the inference baseline is saved.

Choose one route:

- use the supplied checkpoint unchanged;
- freeze the Transformer and train only the classification head;
- fine-tune a small subset of layers;
- fine-tune the full model with a conservative learning rate.

Before training, record which parameter names are trainable. Run one tiny batch through loss, backward and optimizer step. Confirm intended parameters receive gradients. Keep the best validation checkpoint and verify that reloading it reproduces the same predictions.

Do not spend more than 35 minutes on this part.


In [5]:
# TODO: optional adaptation. Keep the baseline available for comparison.


## Part 5 - Token Rationales

For the first 200 validation rows, select at most three token positions that support the model's predicted class.

Requirements:

- indices refer to tokenizer positions, not words in the raw string;
- exclude padding and special tokens;
- preserve the model's original prediction and probability;
- mask only the selected rationale positions and recompute that class probability;
- report mean probability decrease as the faithfulness score;
- decode selected tokens for ten examples and inspect them manually.

You may use input gradients, Integrated Gradients, systematic masking or another method. State which method you chose and why.


In [6]:
# TODO: generate rationale token indices and calculate the faithfulness proxy.


## Part 6 - Submission Contracts

Create two files:

### `submission.csv`

Columns:

- `idx`: exactly the test `idx` values, in test-row order;
- `label`: integer `0` or `1`;
- `positive_probability`: finite float in `[0, 1]`.

### `rationales.jsonl`

Exactly 200 JSON objects, one per line, for the first 200 validation rows:

```json
{"idx": 123, "predicted_label": 1, "token_indices": [4, 5], "original_probability": 0.91, "masked_probability": 0.62}
```

`token_indices` must contain one to three unique integers.


In [7]:
def validate_submission(frame, reference):
    assert frame.columns.tolist() == ["idx", "label", "positive_probability"]
    assert len(frame) == len(reference)
    assert frame["idx"].tolist() == reference["idx"].tolist()
    assert frame["label"].isin([0, 1]).all()
    assert np.isfinite(frame["positive_probability"]).all()
    assert frame["positive_probability"].between(0.0, 1.0).all()


def validate_rationales(path, expected_rows=200):
    records = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]
    assert len(records) == expected_rows
    required = {
        "idx", "predicted_label", "token_indices",
        "original_probability", "masked_probability",
    }
    for record in records:
        assert set(record) == required
        assert record["predicted_label"] in [0, 1]
        assert 1 <= len(record["token_indices"]) <= 3
        assert len(record["token_indices"]) == len(set(record["token_indices"]))
        assert all(isinstance(i, int) and i >= 0 for i in record["token_indices"])
        assert 0.0 <= record["original_probability"] <= 1.0
        assert 0.0 <= record["masked_probability"] <= 1.0
    return records


# TODO: save submission.csv and rationales.jsonl, then call both validators.


## Hints - Open Only If Blocked

1. Hugging Face tokenizers return a dictionary-like batch. Move every tensor in that dictionary to the model device.
2. Sequence-classification outputs expose logits; probabilities require a softmax across the class dimension.
3. Token positions come from `input_ids`. Use the tokenizer's special-token mask or IDs to exclude special tokens.
4. To attribute a class score through embeddings, you need gradients with respect to an input representation while keeping the target scalar differentiable.
5. The faithfulness calculation must compare the same class before and after masking, even if the masked prediction changes class.


## Completion Checklist

The exercise is complete only when all boxes are true:

- [ ] Data schema, split sizes and label values recorded.
- [ ] Two-sentence tokenizer/model API probe saved.
- [ ] Input keys/shapes and output keys/shapes recorded.
- [ ] Baseline validation accuracy saved.
- [ ] Ten validation errors inspected.
- [ ] Trainable parameter names/count recorded if adaptation was attempted.
- [ ] Rationale method and faithfulness score saved.
- [ ] `submission.csv` passes its validator.
- [ ] `rationales.jsonl` passes its validator.
- [ ] Model save/reload reproduces predictions if training was used.

**Stop condition:** after two hours, keep the best format-valid artifacts even if adaptation or rationales are incomplete. Record the exact blocker rather than extending the timebox.
